# Ch 18　測試 LangGraph agent

> **本 notebook 完全不需要 API key**——測試本來就該把 LLM mock 掉。這裡把測試寫成函數直接呼叫（notebook 裡不跑 pytest CLI），通過就印 ✅。

In [ ]:
!uv pip install -q langgraph

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# 起手式：把建圖寫成函數，每個測試重新建、用新的 checkpointer 編譯（互不汙染）
def create_graph() -> StateGraph:
    class MyState(TypedDict):
        my_key: str
    g = StateGraph(MyState)
    g.add_node("node1", lambda s: {"my_key": "hello from node1"})
    g.add_node("node2", lambda s: {"my_key": "hello from node2"})
    g.add_edge(START, "node1"); g.add_edge("node1", "node2"); g.add_edge("node2", END)
    return g

## 18.2 / 18.3 / 18.4　端到端、單節點、部分路徑

In [ ]:
def test_basic_execution():
    graph = create_graph().compile(checkpointer=MemorySaver())
    result = graph.invoke({"my_key": "initial"}, {"configurable": {"thread_id": "1"}})
    assert result["my_key"] == "hello from node2"
    print("✅ 端到端：跑到 node2")

def test_individual_node():
    graph = create_graph().compile(checkpointer=MemorySaver())
    # graph.nodes 暴露每個節點，可單獨 invoke（繞過 checkpointer，純測節點邏輯）
    result = graph.nodes["node1"].invoke({"my_key": "initial"})
    assert result["my_key"] == "hello from node1"
    print("✅ 單節點：node1 邏輯正確")

def test_partial_execution():
    graph = create_graph().compile(checkpointer=MemorySaver())
    config = {"configurable": {"thread_id": "2"}}
    graph.update_state(config, {"my_key": "set by node1"}, as_node="node1")  # 假裝 node1 跑完
    result = graph.invoke(None, config, interrupt_after=["node2"])           # 只往下跑到 node2 就停
    assert result["my_key"] == "hello from node2"
    print("✅ 部分路徑：從 node1 之後跑到 node2")

test_basic_execution(); test_individual_node(); test_partial_execution()

## 18.5　Mock 掉 LLM：測「你的圖邏輯」，不是「模型聰不聰明」

In [ ]:
from typing import Literal

# 一個會「分類 → 路由」的小圖；分類節點在線上版會呼叫 LLM。
class MailState(TypedDict):
    email: str
    category: str
    reply: str

def make_mail_graph(classify_fn):   # 把 classify 當參數注入，方便測試替換成假的
    def route(s) -> Literal["draft", "escalate"]:
        return "draft" if s["category"] == "simple" else "escalate"
    g = (StateGraph(MailState)
         .add_node("classify", classify_fn)
         .add_node("draft", lambda s: {"reply": "[草稿] 已回覆"})
         .add_node("escalate", lambda s: {"reply": "[升級] 轉真人"})
         .add_edge(START, "classify")
         .add_conditional_edges("classify", route, {"draft": "draft", "escalate": "escalate"})
         .add_edge("draft", END).add_edge("escalate", END))
    return g.compile()

def test_complex_routes_to_escalate():
    # mock 掉 LLM 分類：固定回 complex，穩定測「complex 會走 escalate」這條路
    fake_classify = lambda s: {"category": "complex"}
    graph = make_mail_graph(fake_classify)
    result = graph.invoke({"email": "服務一直壞掉！", "category": "", "reply": ""})
    assert result["reply"] == "[升級] 轉真人"
    print("✅ 路由：complex → escalate（LLM 已 mock，測試快又穩）")

test_complex_routes_to_escalate()
print("\n全部測試通過 🎉")

## 重點回顧

每個測試重建圖 + 新 checkpointer；`graph.nodes["x"].invoke` 測單節點；`update_state(as_node=)` + `interrupt_after` 測部分路徑；**LLM 一律 mock**。測試驗證「圖邏輯對不對」，LLM 輸出「好不好」交給 eval（Ch36）。